In [15]:
import re
import math
import os
import pandas as pd

# =============================================================================
# 1. Data Parser
# =============================================================================
def parse_dat_file(dat_filepath):
    # Defaults matching the OPL model
    params = {
        'scaleLength': 648.0,  # 648mm (ギターのスケール長)
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 22, 'p_max': 22,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.5, 2.0, 3.0],
        'natural_min_spread_mm': [20.0, 40.0, 55.0],  # OPL: knuckleDist[2,3,4]
        'availableTime': [],
        'neck_thickness': 1,  # 整数モード (1, 2, 3)
        # Weights (AC削除)
        'd_PC': 1.5, 'd_SC': 1.0, 'd_ST': 1.0, 'd_DH': 1.0, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2
    }

    print(f"DEBUG: Parsing .dat file: {dat_filepath}")
    if not os.path.exists(dat_filepath):
        print("WARNING: File not found. Using defaults.")
        return params

    try:
        content = ""
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(dat_filepath, 'r', encoding=enc) as f:
                    content = f.read()
                break
            except UnicodeDecodeError: continue

        def get_val(name, cast_func):
            pat = re.compile(rf'{name}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: return cast_func(m.group(1))
            return None

        def get_arr(name):
            pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
            m = pat.search(content)
            if m:
                txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
                return [float(v) for v in txt.split(',') if v.strip()]
            return None

        for k in params.keys():
            if isinstance(params[k], list):
                val = get_arr(k)
                if val: params[k] = val
            else:
                val = get_val(k, float)
                if val is not None: params[k] = val
        
        params['s_max'] = int(params['s_max'])
        params['f_max'] = int(params['f_max'])
        params['neck_thickness'] = int(params.get('neck_thickness', 1))
        
        return params
    except Exception as e:
        print(f"ERROR: Parsing failed ({e}). Using defaults.")
        return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath): return []
    
    try:
        lines = []
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(solution_filepath, 'r', encoding=enc) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError: continue

        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue
            if not header_found:
                if "Note" in line and "String" in line: header_found = True
                continue
            if line.startswith('---'): break
            
            parts = re.split(r'\s+', line)
            if parts[0] == "Note": continue
            if len(parts) >= 5:
                try:
                    if parts[0] == 'R':
                         note = {'note': 'R', 'string': 0, 'fret': 0, 'hand': 0, 'finger': 0}
                    else:
                        note = {
                            'note': parts[0],
                            'string': int(parts[1]),
                            'fret': int(parts[2]),
                            'hand': int(parts[3]),
                            'finger': 0 if parts[4] == 'x' else int(parts[4])
                        }
                    notes_data.append(note)
                except ValueError: continue
        return notes_data
    except: return []

# =============================================================================
# 2. OPL Model Reproducer (modified_model.mod 完全準拠版)
# =============================================================================
class GuitarModelReproducer:
    def __init__(self, params):
        self.p = params
        
        # --- OPL準拠パラメータ設定 ---
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']  # [0, 70, 75, 70, 50]
        self.fingerDiff = [0.0] + self.p['fingerDifficulty']  # [0, 1.0, 1.5, 2.0, 3.0]
        self.knuckleDist = [0.0, 0.0] + self.p['natural_min_spread_mm']  # [0, 0, 20, 40, 55]
        
        # limitRatios: OPL準拠
        self.limitRatios = [0.0, 1.3, 1.05, 1.05, 1.1]
        
        # baseFactors: OPL準拠 (Index=0.7, Middle=0.9, Ring=0.85, Pinky=1.0)
        self.baseFactors = [0.0, 0.7, 0.9, 0.85, 0.95]
        
        # finger_mult: OPL準拠 (Index/Middle=0.0, Ring=1.2, Pinky=1.3)
        self.finger_mult = [0.0, 0.0, 0.0, 1.2, 1.3]
        
        # Neck Factor Parameters: OPL準拠
        neck_thickness = self.p.get('neck_thickness', 1)
        if neck_thickness == 1:
            self.max_reduction_at_p16 = 0.15
        elif neck_thickness == 2:
            self.max_reduction_at_p16 = 0.225
        elif neck_thickness == 3:
            self.max_reduction_at_p16 = 0.30
        else:
            self.max_reduction_at_p16 = 0.15  # default
        
        self.target_reduction_at_p12 = 0.03
        self.p_max_ref = 16.0
        
        # curve_exponent k = ln(0.03 / Max) / ln(12 / 16)
        if self.max_reduction_at_p16 > self.target_reduction_at_p12:
            self.curve_exponent = math.log(self.target_reduction_at_p12 / self.max_reduction_at_p16) / math.log(12.0 / 16.0)
        else:
            self.curve_exponent = 1.0
        
        # Physics用物理定数 (Grimes 2014 Model)
        self.E_GPa = {1: 179.0, 2: 188.0, 3: 177.0, 4: 62.0, 5: 42.0, 6: 33.0}
        self.Area_m2 = {1: 5.13e-8, 2: 8.56e-8, 3: 1.48e-7, 4: 3.55e-7, 5: 6.79e-7, 6: 1.10e-6}
        self.Tension_N = {1: 73.0, 2: 70.0, 3: 68.0, 4: 75.0, 5: 78.0, 6: 76.0}
        
        # 定数
        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        
        # フレット位置計算
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]
        
        # Time penalty parameters
        self.tp_base = 0.15
        self.tp_ref = 60.0
        self.tp_exp = 1.5

    def calculate_neck_factor(self, hand_pos, finger_idx):
        """OPL準拠: ポジション基準のNeck Factor計算
        
        reduction = max_reduction * (pos / 16)^k * finger_mult[h]
        neckFactor = 1.0 - reduction (min 0.1)
        """
        current_pos = hand_pos
        ratio = 0.0
        
        if current_pos >= 0:
            ratio = current_pos / self.p_max_ref
        if ratio > 1.0:
            ratio = 1.0
        
        # 減少率 = Max * (ratio ^ k) * finger_mult
        reduction = self.max_reduction_at_p16 * (ratio ** self.curve_exponent) * self.finger_mult[finger_idx]
        
        # 係数 (1.0 - 減少分), 最低0.1
        neckFactor = 1.0 - reduction
        if neckFactor < 0.1:
            neckFactor = 0.1
        
        return neckFactor

    def calculate_stretch_cost(self, hand_pos, finger_idx, string, fret):
        """OPL準拠: 角度探索 (-90° to 0°) によるStretch Cost計算"""
        if fret < hand_pos:
            return 100000.0
        if fret > self.p['f_max']:
            return 100000.0
        
        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0
        
        minStCost = 100000.0
        foundValidAngle = False
        
        for ang in range(-90, 1, 2):
            thetaRad = ang * self.deg_to_rad
            
            d = self.knuckleDist[finger_idx] if finger_idx < len(self.knuckleDist) else 0.0
            kX = pivotX + (d * math.cos(thetaRad))
            kY = pivotY + (d * math.sin(thetaRad))
            
            tX = self.fret_pos[fret]
            tY = (string - 1) * self.p['stringSpacing']
            
            vX = tX - kX
            vY = tY - kY
            dist = math.sqrt(vX*vX + vY*vY)
            
            # OPL準拠: Neck Factorはhand_posとfinger_idxに依存
            neckFactor = self.calculate_neck_factor(hand_pos, finger_idx)
            effLen = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] * neckFactor
            
            if effLen > 0 and dist > effLen * self.limitRatios[finger_idx]:
                continue
            
            nX = -math.sin(thetaRad)
            nY = math.cos(thetaRad)
            
            dotProd = vX * nX + vY * nY
            cosDelta = (dotProd / dist) if dist > 0.001 else 1.0
            cosDelta = max(-1.0, min(1.0, cosDelta))
            deltaAbsDeg = math.acos(cosDelta) * self.rad_to_deg
            
            crossProd = nX * vY - nY * vX
            isLowFretSide = (crossProd > 0)
            deltaSignedDeg = deltaAbsDeg if isLowFretSide else -deltaAbsDeg
            
            if isLowFretSide:
                limitAngle = 15.0
                relaxAdd = 5.0
            else:
                limitAngle = 30.0
                relaxAdd = 10.0
            
            if finger_idx == 4:
                limitAngle += 5.0
            
            dist_ratio = 0.0
            isCloseDist = False
            if effLen > 0:
                dist_ratio = dist / effLen
                if dist_ratio < 0.5:
                    limitAngle += relaxAdd
                    isCloseDist = True
            
            if deltaAbsDeg > limitAngle:
                continue
            
            # OPL準拠: distSlope は一律 20.0
            distCost = 0.0
            distSlope = 20.0
            
            if dist_ratio < 0.3:
                distCost = (0.3 - dist_ratio) * 5.0
            elif dist_ratio <= 0.6:
                distCost = 0.0
            else:
                distCost = (dist_ratio - 0.6) * distSlope
                if dist_ratio > 1.0:
                    distCost += (dist_ratio - 1.0) * 30.0
            
            comfortAngle = 5.0
            angleCost = 0.0
            coeffPositive = 1.5
            coeffNegative = 1.0
            
            if deltaSignedDeg > comfortAngle:
                angleCost = (deltaSignedDeg - comfortAngle) * coeffPositive
            elif deltaSignedDeg < -comfortAngle:
                angleCost = (abs(deltaSignedDeg) - comfortAngle) * coeffNegative
            
            if isCloseDist:
                angleCost *= 0.5
            
            currentCost = distCost + angleCost
            
            if currentCost < minStCost:
                minStCost = currentCost
                foundValidAngle = True
        
        if foundValidAngle:
            return minStCost
        else:
            return 100000.0

    def calculate_physics_cost(self, string, fret):
        """Grimes (2014) Eq.7 に基づく物理コスト計算"""
        if fret > self.p['f_max']:
            return 0.0
        
        a_mm = 0.5 + (2.0 * fret / 12.0)
        l_mm = self.p['scaleLength'] - self.fret_pos[fret]
        
        if l_mm <= 0.001:
            return 0.0
        
        l_m = l_mm / 1000.0
        a_m = a_mm / 1000.0
        
        E = self.E_GPa[string] * 1.0e9
        A = self.Area_m2[string]
        T = self.Tension_N[string]
        
        hyp_m = math.sqrt(l_m * l_m + a_m * a_m)
        sin_theta = a_m / hyp_m
        cos_theta = l_m / hyp_m
        
        strain_term = (1.0 / cos_theta) - 1.0
        F_newton = (T + (E * A * strain_term)) * sin_theta
        
        scalingFactor = 15.0
        return F_newton * scalingFactor

    def calculate_total(self, notes_data):
        total = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
        details = []
        num_notes = len(notes_data)
        avail_time = self.p['availableTime']
        if len(avail_time) < num_notes:
            avail_time += [30.0] * (num_notes - len(avail_time))
        
        # --- 前処理: prevNoteIndex, accumulatedTime 計算 ---
        prev_idx_map = {}
        acc_time = {}
        
        for i in range(num_notes):
            is_rest = (notes_data[i]['note'] == 'R')
            if not is_rest:
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                if p >= 0:
                    prev_idx_map[i] = p
                    acc_time[i] = t_sum + avail_time[p]
                else:
                    prev_idx_map[i] = i
                    acc_time[i] = 10000.0
            else:
                acc_time[i] = 0.0
                prev_idx_map[i] = -1
        
        # --- コスト事前計算 ---
        stretch_costs = []
        physics_costs = []
        
        for i in range(num_notes):
            curr = notes_data[i]
            if curr['note'] == 'R' or curr['fret'] == 0:
                stretch_costs.append(0.0)
                physics_costs.append(0.0)
            else:
                finger_idx = curr['finger'] if curr['finger'] in [1, 2, 3, 4] else 1
                hand_pos = curr['hand']
                
                st_cost = self.calculate_stretch_cost(hand_pos, finger_idx, curr['string'], curr['fret'])
                ph_cost = self.calculate_physics_cost(curr['string'], curr['fret'])
                
                diff = self.fingerDiff[finger_idx]
                ph_cost *= diff
                
                stretch_costs.append(st_cost if st_cost < 50000 else 100000.0)
                physics_costs.append(ph_cost)
        
        # --- メインループ: コスト計算 ---
        for i in range(num_notes):
            curr = notes_data[i]
            row = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
            time_penalty = 0.0
            
            if curr['note'] != 'R':
                prev_i = prev_idx_map.get(i, i)
                
                if prev_i == i:
                    time_penalty = 0.0
                else:
                    t_val = acc_time[i] + 0.01
                    time_penalty = self.tp_base + (self.tp_ref / t_val) ** self.tp_exp
                
                if curr['fret'] == 0:
                    row['NO'] = 1.0
                
                if curr['finger'] in [1, 2, 3, 4] and curr['fret'] > 0:
                    row['ST'] = stretch_costs[i]
                    row['DH'] = physics_costs[i]
                
                if prev_i != i and prev_i != -1:
                    prev = notes_data[prev_i]
                    row['PC'] = abs(self.fret_pos[prev['hand']] - self.fret_pos[curr['hand']])
                    row['SC'] = self.p['stringSpacing'] * abs(prev['string'] - curr['string'])
            
            # --- 重み付きコスト計算 ---
            w_pc = row['PC'] * time_penalty * self.p['d_PC'] * self.p['alpha_PC']
            w_sc = row['SC'] * time_penalty * self.p['d_SC'] * self.p['alpha_SC']
            w_st = row['ST'] * self.p['d_ST'] * self.p['alpha_ST']
            w_dh = row['DH'] * self.p['d_DH'] * self.p['alpha_DH']
            w_no = row['NO'] * self.p['d_NO'] * self.p['alpha_NO']
            
            total['PC'] += w_pc
            total['SC'] += w_sc
            total['ST'] += w_st
            total['DH'] += w_dh
            total['NO'] += w_no
            
            sum_weighted = w_pc + w_sc + w_st + w_dh + w_no
            
            details.append({
                'Index': i + 1, 'Note': curr['note'],
                'Time(ms)': acc_time.get(i, 0),
                'TimeFactor': time_penalty,
                'PC_w': w_pc, 'SC_w': w_sc, 'ST_w': w_st, 'PH_w': w_dh, 'NO_w': w_no,
                'Total': sum_weighted
            })
        
        grand_total = sum(total.values())
        return total, grand_total, pd.DataFrame(details)

In [19]:
# --- 実行・表示部分 ---
if __name__ == "__main__":
    dat_file = "datas/complex_solo_mod.dat"
    sol_file = "complex_opt.txt"
    
    if os.path.exists(dat_file) and os.path.exists(sol_file):
        params = parse_dat_file(dat_file)
        
        # --- .datファイルから読み込んだパラメータの表示 ---
        print("\n" + "="*60)
        print(" --- OPL Model Parameters (from .dat file) ---")
        print(f"  scaleLength     : {params['scaleLength']} mm")
        print(f"  stringSpacing   : {params['stringSpacing']} mm")
        print(f"  f_max           : {params['f_max']}")
        print(f"  neck_thickness  : {params.get('neck_thickness', 1)}")
        print(f"  fingerLengths   : {params['fingerLengths_mm']}")
        print(f"  fingerDifficulty: {params['fingerDifficulty']}")
        print(f"  natural_spread  : {params['natural_min_spread_mm']}")
        print("-" * 40)
        print(f"  d_PC={params['d_PC']}, d_SC={params['d_SC']}, d_ST={params['d_ST']}")
        print(f"  d_DH={params['d_DH']}, d_NO={params['d_NO']}")
        print(f"  alpha_PC={params['alpha_PC']}, alpha_SC={params['alpha_SC']}, alpha_ST={params['alpha_ST']}")
        print(f"  alpha_DH={params['alpha_DH']}, alpha_NO={params['alpha_NO']}")
        print("="*60)
        
        # --- ハードコードパラメータ表示 (OPL準拠) ---
        print("\n --- Hardcoded Parameters (OPL model) ---")
        print("  limitRatios  : [0, 1.3, 1.05, 1.05, 1.1]  (Index, Middle, Ring, Pinky)")
        print("  baseFactors  : [0, 0.7, 0.9, 0.85, 0.95]   (Index, Middle, Ring, Pinky)")
        print("  finger_mult  : [0, 0.0, 0.0, 1.2, 1.3]    (NeckFactor multiplier)")
        print("  knuckleDist  : [0, 0, 20, 40, 55]         (from natural_min_spread_mm)")
        print("  distSlope    : 20.0 (uniform)")
        print("  Physics: Grimes 2014 Eq.7 (E, A, T per string)")
        print("  Angle Search: -90° to 0° (step 2°)")
        print("  Neck Factor : Position-based (p/16)^k * finger_mult")
        print("="*60)
        
        model = GuitarModelReproducer(params)
        
        # Neck Factor情報表示
        print(f"\n --- Neck Factor Info ---")
        print(f"  neck_thickness mode: {params.get('neck_thickness', 1)}")
        print(f"  max_reduction_at_p16: {model.max_reduction_at_p16}")
        print(f"  curve_exponent k: {model.curve_exponent:.4f}")
        print("="*60)
        
        notes = parse_solution_file(sol_file)
        
        if notes:
            bd, total, df = model.calculate_total(notes)
            print("\n" + "="*60)
            print(" --- Cost Breakdown ---")
            print(f"  Position Change (PC): {bd['PC']:.4f}")
            print(f"  String Change   (SC): {bd['SC']:.4f}")
            print(f"  Geo Stretch     (ST): {bd['ST']:.4f}")
            print(f"  Phys Action     (DH): {bd['DH']:.4f}")
            print(f"  Open String     (NO): {bd['NO']:.4f}")
            print("-" * 40)
            print(f"  Total Cost (Sum)    : {total:.4f}")
            print("="*60)
            if not df.empty:
                print("\n[Details - All Notes]")
                pd.set_option('display.max_columns', None)
                pd.set_option('display.width', 1000)
                pd.set_option('display.max_rows', None)
                print(df[['Index', 'Note', 'Time(ms)', 'TimeFactor', 'PC_w', 'SC_w', 'ST_w', 'PH_w', 'NO_w', 'Total']].to_string(index=False, float_format="%.3f"))
    else:
        print("Please upload 'datas/complex_solo_mod.dat' and solution file to run the calculation.")

DEBUG: Parsing .dat file: datas/complex_solo_mod.dat

 --- OPL Model Parameters (from .dat file) ---
  scaleLength     : 628.0 mm
  stringSpacing   : 10.8 mm
  f_max           : 22
  neck_thickness  : 3
  fingerLengths   : [70.0, 75.0, 70.0, 50.0]
  fingerDifficulty: [1.0, 1.2, 1.3, 1.8]
  natural_spread  : [20.0, 40.0, 55.0]
----------------------------------------
  d_PC=0.9, d_SC=0.7, d_ST=1.5
  d_DH=0.5, d_NO=10.0
  alpha_PC=0.2, alpha_SC=0.2, alpha_ST=0.2
  alpha_DH=0.2, alpha_NO=0.2

 --- Hardcoded Parameters (OPL model) ---
  limitRatios  : [0, 1.3, 1.05, 1.05, 1.1]  (Index, Middle, Ring, Pinky)
  baseFactors  : [0, 0.7, 0.9, 0.85, 0.95]   (Index, Middle, Ring, Pinky)
  finger_mult  : [0, 0.0, 0.0, 1.2, 1.3]    (NeckFactor multiplier)
  knuckleDist  : [0, 0, 20, 40, 55]         (from natural_min_spread_mm)
  distSlope    : 20.0 (uniform)
  Physics: Grimes 2014 Eq.7 (E, A, T per string)
  Angle Search: -90° to 0° (step 2°)
  Neck Factor : Position-based (p/16)^k * finger_mult

 -

In [11]:
# デバッグ: Sumの確認
print("bd (breakdown dict):", bd)
print("total (grand_total):", total)
print()
print("Sum of bd.values():", sum(bd.values()))
print()
print("df['Total'].sum():", df['Total'].sum())

bd (breakdown dict): {'PC': 92.0503426585457, 'SC': 64.93538353092532, 'ST': 76.34899848611404, 'DH': 152.3392062484138, 'NO': 0.0}
total (grand_total): 385.67393092399885

Sum of bd.values(): 385.67393092399885

df['Total'].sum(): 385.67393092399885
